In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as si
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")
import QuantLib as ql
import plotly.graph_objects as go
import os
import regex as re
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy import interpolate
from scipy.optimize import minimize
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, timedelta

In [ ]:
data_path = "./putcall.csv"

tmp = pd.read_csv(data_path, parse_dates=['QUOTE_DATE', 'EXPIRE_DATE'])

# 确保 QUOTE_DATE 列为 datetime64 类型
tmp['QUOTE_DATE'] = pd.to_datetime(tmp['QUOTE_DATE'])

# 筛选 2023 年的数据
tmp = tmp[tmp['QUOTE_DATE'].dt.year == 2023]

tmp = tmp[tmp['DTE'] == 365 * 2].reset_index(drop = True)

tmp

# merge the daily SSVI parameter on “tmp” dataframe

In [ ]:
ssvi_para = pd.read_csv("./ssvi_para.csv")


# 合并DataFrame
ssvi_para.rename(columns={'date': 'QUOTE_DATE'}, inplace=True)

ssvi_para['QUOTE_DATE'] = pd.to_datetime(ssvi_para['QUOTE_DATE'])

ssvi_para = ssvi_para[ssvi_para['QUOTE_DATE'] == '2023-06-21']


df_merged_tmp_ssvi = pd.merge(tmp, ssvi_para, on='QUOTE_DATE', how='left')


df_merged_tmp_ssvi

# calculate implied volatility based on SSVI

In [ ]:
def compute_theta_t(params, t, kappa1, kappa2):
    V, V_prime, theta = params
    theta_t = theta * t + (V - theta) * (1 - np.exp(-kappa1 * t)) / kappa1 + \
              (V_prime - theta) * kappa1 * ((1 - np.exp(-kappa2 * t)) / kappa2 - \
              (1 - np.exp(-kappa1 * t)) / kappa1) / (kappa1 - kappa2)
    return theta_t

def compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    if phi_type == "POWER-LAW1":
        return eta / theta_t ** gamma
    elif phi_type == "POWER-LAW2":
        return eta / (theta_t ** gamma * (1 + theta_t) ** (1 - gamma))
    elif phi_type == "SQRT":
        return eta / np.sqrt(theta_t)
    elif phi_type == "HESTON":
        return (1 - (1 - np.exp(-gamma * theta_t)) / (gamma * theta_t)) / (gamma * theta_t)
    else:
        raise ValueError("Invalid phi_type")

def compute_ssvi_inference(params_w_t, params_theta_t, t, k, phi_type, kappa1, kappa2):
    rho, eta, gamma = params_w_t
    theta_t = compute_theta_t(params_theta_t, t, kappa1, kappa2)
    phi = compute_phi(params_w_t, params_theta_t, t, phi_type, kappa1, kappa2)
    return 0.5 * theta_t * (1 + rho * phi * k + np.sqrt((phi * k + rho) ** 2 + 1 - rho ** 2))


# 定义 kappa1 和 kappa2
kappa1 = 5.5
kappa2 = 0.1

# 使用 apply 方法，传递所有必要参数
df_merged_tmp_ssvi['ssvi_iv'] = df_merged_tmp_ssvi.apply(
    lambda row: np.sqrt(
        compute_ssvi_inference(
            params_w_t=[row['rho'], row['eta'], row['gamma']],
            params_theta_t=[row['V'], row['V_prime'], row['theta']],
            t=row['DTE'] / 365,
            k=row['Log-Strike'],
            phi_type="POWER-LAW1",
            kappa1=kappa1,
            kappa2=kappa2
        ) / (row['DTE'] / 365)
    ), axis=1
)

df_merged_tmp_ssvi

In [ ]:
import numpy as np
    
def binomial_tree_american_option_vectorized(S0, K, T, r, sigma, N, option_type="call"):
    
    """
    Prices an American option using the binomial tree model.
    
    Parameters:
    S0 : float - Initial stock price
    K : float - Strike price
    T : float - Time to maturity in years
    r : float - Risk-free interest rate
    sigma : float - Volatility
    N : int - Number of steps in the binomial tree
    option_type : str - Type of the option ("call" or "put")
    
    Returns:
    float - Estimated option price
    """
    
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)

    # 初始化股票价格树
    stock_prices = np.asarray([S0 * u**j * d**(N-j) for j in range(N+1)])
    
    # 初始化期权价值树的最终节点
    if option_type == "call":
        option_values = np.maximum(stock_prices - K, 0)
    else:  # put
        option_values = np.maximum(K - stock_prices, 0)

    # 反向迭代期权价值树
    for i in range(N-1, -1, -1):
        stock_prices = S0 * u**np.arange(i+1) * d**(i-np.arange(i+1))
        option_values_early = np.maximum(stock_prices - K, 0) if option_type == "call" else np.maximum(K - stock_prices, 0)
        option_values = np.exp(-r * dt) * (p * option_values[1:] + (1 - p) * option_values[:-1])
        option_values = np.maximum(option_values, option_values_early)  # 早期行权的检查

    return option_values[0]



# 设置二叉树的步数
N = 100  # 或者根据需要调整

# 定义计算期权价格的函数
def calc_option_price(row, option_type="call"):
    return binomial_tree_american_option(
        S0=row['UNDERLYING_LAST'], 
        K=row['STRIKE'], 
        T=row['DTE'] / 365,  # 假设DTE是以天为单位，转换为年
        r=row['rf'], 
        sigma=row['ssvi_iv'], 
        N=N,
        option_type=option_type
    )

# 计算每一行的美式看涨期权和看跌期权价格
df_merged_tmp_ssvi['American_Call_Price'] = df_merged_tmp_ssvi.apply(calc_option_price, option_type="call", axis=1)
df_merged_tmp_ssvi['American_Put_Price'] = df_merged_tmp_ssvi.apply(calc_option_price, option_type="put", axis=1)

# 从 df_merged_tmp_ssvi DataFrame 中提取 'American_Call_Price' 和 'American_Put_Price' 列
df_prices = df_merged_tmp_ssvi[['American_Call_Price', 'American_Put_Price']]

df_prices